# Russian ASR ITN Hybrid

Обратная текстовая нормализация (ITN) для русского языка.

**Два параллельных пути:**
1. **Legacy parser**: `normalize_text()` -> `parse_number_group()`
2. **Sequence parser**: `normalize_text_sequence()` -> `classify()` -> `parse_sequence()`

In [ ]:
import sys, os, re, inspect, collections
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
import pandas as pd
import pyarrow.feather as feather
import numpy as np

---

## 1. Словари числительных

In [ ]:
UNITS = {
    "ноль": (0, 0),
    "один": (1, 0), "одна": (1, 0), "одно": (1, 0),
    "одного": (1, 0), "одной": (1, 0), "одному": (1, 0),
    "одним": (1, 0), "одном": (1, 0),
    "одни": (1, 0), "одних": (1, 0),
    "два": (2, 0), "две": (2, 0),
    "двух": (2, 0), "двум": (2, 0), "двумя": (2, 0),
    "три": (3, 0), "трёх": (3, 0), "трех": (3, 0), "трем": (3, 0), "тремя": (3, 0),
    "четыре": (4, 0),
    "четырёх": (4, 0), "четырех": (4, 0),
    "четырем": (4, 0), "четырьмя": (4, 0),
    "пять": (5, 0), "пяти": (5, 0), "пятью": (5, 0),
    "шесть": (6, 0), "шести": (6, 0), "шестью": (6, 0),
    "семь": (7, 0), "семи": (7, 0), "семью": (7, 0),
    "восемь": (8, 0), "восьми": (8, 0), "восемью": (8, 0),
    "девять": (9, 0), "девяти": (9, 0), "девятью": (9, 0),
}

In [ ]:
COLLECTIVES = {
    "двое": (2, 0), "трое": (3, 0), "четверо": (4, 0),
    "пятеро": (5, 0), "шестеро": (6, 0), "семеро": (7, 0),
    "восьмеро": (8, 0), "девятеро": (9, 0), "десятеро": (10, 0),
}

In [ ]:
TEENS = {
    "десять": (10, 1), "десяти": (10, 1), "десятью": (10, 1),
    "одиннадцать": (11, 1), "одиннадцати": (11, 1),
    "двенадцать": (12, 1), "двенадцати": (12, 1),
    "тринадцать": (13, 1), "тринадцати": (13, 1),
    "четырнадцать": (14, 1), "четырнадцати": (14, 1),
    "пятнадцать": (15, 1), "пятнадцати": (15, 1),
    "шестнадцать": (16, 1), "шестнадцати": (16, 1),
    "семнадцать": (17, 1), "семнадцати": (17, 1),
    "восемнадцать": (18, 1), "восемнадцати": (18, 1),
    "девятнадцать": (19, 1), "девятнадцати": (19, 1),
}

In [ ]:
TENS = {
    "двадцать": (20, 2), "двадцати": (20, 2),
    "тридцать": (30, 2), "тридцати": (30, 2), "тридцатью": (30, 2),
    "сорок": (40, 2), "сорока": (40, 2),
    "пятьдесят": (50, 2), "пятидесяти": (50, 2),
    "шестьдесят": (60, 2), "шестидесяти": (60, 2),
    "семьдесят": (70, 2), "семидесяти": (70, 2),
    "восемьдесят": (80, 2), "восьмидесяти": (80, 2),
    "девяносто": (90, 2), "девяноста": (90, 2),
}

In [ ]:
HUNDREDS = {
    "сто": (100, 3), "ста": (100, 3),
    "двести": (200, 3),
    "двухсот": (200, 3), "двумстам": (200, 3),
    "двумястами": (200, 3), "двухстах": (200, 3),
    "двеси": (200, 3), "дваста": (200, 3),
    "триста": (300, 3),
    "трёхсот": (300, 3), "трехсот": (300, 3),
    "тремстам": (300, 3), "тремястами": (300, 3), "трёхстах": (300, 3),
    "четыреста": (400, 3),
    "четырёхсот": (400, 3), "четырехсот": (400, 3),
    "четыремстам": (400, 3), "четырьмястами": (400, 3),
    "пятьсот": (500, 3), "пятисот": (500, 3),
    "шестьсот": (600, 3), "шестисот": (600, 3),
    "семьсот": (700, 3), "семисот": (700, 3),
    "восемьсот": (800, 3), "восьмисот": (800, 3),
    "девятьсот": (900, 3), "девятисот": (900, 3),
}

In [ ]:
THOUSANDS = {
    "тысяча": (1000, 4), "тысячи": (1000, 4), "тысяч": (1000, 4),
    "тысячу": (1000, 4), "тысяче": (1000, 4), "тысячей": (1000, 4),
    "тысячам": (1000, 4), "тысячами": (1000, 4), "тысячах": (1000, 4),
    "тыщ": (1000, 4), "тыщи": (1000, 4), "тыща": (1000, 4),
}
MILLIONS = {
    "миллион": (1000000, 5), "миллиона": (1000000, 5),
    "миллионов": (1000000, 5), "миллионом": (1000000, 5),
    "миллионам": (1000000, 5), "миллионами": (1000000, 5), "миллионах": (1000000, 5),
}
BILLIONS = {
    "миллиард": (1000000000, 6), "миллиарда": (1000000000, 6),
    "миллиардов": (1000000000, 6),
    "миллиардам": (1000000000, 6), "миллиардами": (1000000000, 6), "миллиардах": (1000000000, 6),
}
MULTIPLIERS = set()
for d in (THOUSANDS, MILLIONS, BILLIONS):
    MULTIPLIERS.update(d.keys())

In [ ]:
ASR_ERRORS = {
    "двеси": "двести", "дваста": "двести",
    "тыщ": "тысяч", "тыщи": "тысячи",
    "петьсят": "пятьдесят", "петьдесят": "пятьдесят",
    "писят": "пятьдесят", "пядсят": "пятьдесят", "пьсят": "пятьдесят",
    "пядцить": "пятнадцать", "пятнадцить": "пятнадцать",
    "питнадцать": "пятнадцать", "петнадцать": "пятнадцать",
    "симнадцить": "семнадцать", "семнадцить": "семнадцать",
    "шиснатцать": "шестнадцать", "шеснатцать": "шестнадцать",
    "восимнадцить": "восемнадцать", "восемнадцить": "восемнадцать",
    "дивяносто": "девяносто", "дивяноста": "девяноста",
    "тысич": "тысяч", "тысеч": "тысяч", "тисяч": "тысяч", "тищ": "тысяч",
    "милион": "миллион", "милиона": "миллиона", "милионов": "миллионов",
    "тристи": "триста", "семсот": "семьсот",
    "восимсот": "восемьсот", "девятсот": "девятьсот",
    "двенацтать": "двенадцать", "тринацать": "тринадцать", "четырнацать": "четырнадцать",
    "восемсат": "восемьдесят", "восимдесат": "восемьдесят",
    "дисять": "десять", "дисят": "десять", "десят": "десять",
    "тыc": "тысяч", "тыс": "тысяч",
}

In [ ]:
NUMERAL_DICT = {}
for d in (UNITS, TEENS, TENS, HUNDREDS, THOUSANDS, MILLIONS, BILLIONS):
    NUMERAL_DICT.update(d)
NUMERAL_DICT.update({"дветысячи": (2000, 0), "двесте": (200, 3), "двестипятьсот": (2500, 0)})

In [ ]:
ORDINALS = {
    "первый": "1", "первого": "1", "первому": "1",
    "первым": "1", "первом": "1",
    "первая": "1", "первой": "1", "первую": "1",
    "первое": "1", "первые": "1", "первых": "1", "первыми": "1",
    "второй": "2", "второго": "2", "второму": "2",
    "вторым": "2", "втором": "2",
    "вторая": "2", "вторую": "2", "второе": "2",
    "вторые": "2", "вторых": "2", "вторыми": "2",
    "третий": "3", "третьего": "3", "третьему": "3",
    "третьим": "3", "третьем": "3",
    "третья": "3", "третьей": "3", "третью": "3",
    "третье": "3", "третьи": "3", "третьих": "3", "третьими": "3",
    "четвертый": "4", "четвертого": "4", "четвертому": "4",
    "четвертым": "4", "четвертом": "4",
    "четвертая": "4", "четвертой": "4", "четвертую": "4", "четвертое": "4",
    "четвертые": "4", "четвертых": "4",
    "пятый": "5", "пятого": "5", "пятому": "5",
    "пятым": "5", "пятом": "5",
    "пятая": "5", "пятой": "5", "пятую": "5", "пятое": "5",
    "пятые": "5", "пятых": "5",
    "шестой": "6", "шестого": "6", "шестому": "6",
    "шестым": "6", "шестом": "6",
    "шестая": "6", "шестую": "6", "шестое": "6",
    "шестые": "6", "шестых": "6",
    "седьмой": "7", "седьмого": "7", "седьмому": "7",
    "седьмым": "7", "седьмом": "7",
    "седьмая": "7", "седьмую": "7", "седьмое": "7",
    "седьмые": "7", "седьмых": "7",
    "восьмой": "8", "восьмого": "8", "восьмому": "8",
    "восьмым": "8", "восьмом": "8",
    "восьмая": "8", "восьмую": "8", "восьмое": "8",
    "восьмые": "8", "восьмых": "8",
    "девятый": "9", "девятого": "9", "девятому": "9",
    "девятым": "9", "девятом": "9",
    "девятая": "9", "девятой": "9", "девятую": "9",
    "девятое": "9", "девятые": "9", "девятых": "9",
    "десятый": "10", "десятого": "10", "десятому": "10",
    "десятым": "10", "десятом": "10",
    "одиннадцатый": "11", "одиннадцатого": "11",
    "одиннадцатому": "11", "одиннадцатым": "11",
    "одиннадцатом": "11", "одиннадцатое": "11",
    "двенадцатый": "12", "двенадцатого": "12",
    "двенадцатому": "12", "двенадцатым": "12",
    "двенадцатом": "12", "двенадцатое": "12",
    "тринадцатый": "13", "тринадцатого": "13",
    "тринадцатому": "13", "тринадцатым": "13",
    "тринадцатом": "13", "тринадцатое": "13",
    "четырнадцатый": "14", "четырнадцатого": "14", "четырнадцатое": "14",
    "пятнадцатый": "15", "пятнадцатого": "15", "пятнадцатое": "15",
    "шестнадцатый": "16", "шестнадцатого": "16", "шестнадцатое": "16",
    "семнадцатый": "17", "семнадцатого": "17", "семнадцатое": "17",
    "восемнадцатый": "18", "восемнадцатого": "18", "восемнадцатое": "18",
    "девятнадцатый": "19", "девятнадцатого": "19", "девятнадцатое": "19",
    "двадцатый": "20", "двадцатого": "20",
    "тридцатый": "30", "тридцатого": "30",
    "сороковой": "40", "сорокового": "40",
    "пятидесятый": "50", "пятидесятого": "50",
    "шестидесятый": "60", "семидесятый": "70", "восьмидесятый": "80",
    "девяностый": "90", "девяностого": "90",
    "сотый": "100", "сотого": "100",
    "двухсотый": "200", "двухсотая": "200", "двухсотую": "200",
    "трёхсотый": "300", "четырехсотый": "400",
    "пятисотый": "500", "шестисотый": "600",
    "семисотый": "700", "восьмисотый": "800", "девятисотый": "900",
}
ORDINAL_SET = set(ORDINALS.keys())

---
## 2. Token Classifier

In [ ]:
@dataclass
class TokenClass:
    value: int
    mag: int
    subtype: str
    raw: str
    confidence: float = 1.0

In [ ]:
_NUMERIC_ROOTS = [
    ("миллиард", 1000000000, 6, True), ("миллион", 1000000, 5, True),
    ("тысяч", 1000, 4, True), ("тыщ", 1000, 4, True),
    ("пятидесят", 50, 2, False), ("пятьдесят", 50, 2, False),
    ("шестидесят", 60, 2, False), ("шестьдесят", 60, 2, False),
    ("семидесят", 70, 2, False), ("семьдесят", 70, 2, False),
    ("восьмидесят", 80, 2, False), ("восемьдесят", 80, 2, False),
    ("двадцат", 20, 2, False), ("тридцат", 30, 2, False),
    ("девяност", 90, 2, False), ("десят", 10, 1, False),
    ("сорок", 40, 2, False),
    ("девятьсот", 900, 3, False), ("девятисот", 900, 3, False),
    ("восемьсот", 800, 3, False), ("восьмисот", 800, 3, False),
    ("семьсот", 700, 3, False), ("семисот", 700, 3, False),
    ("шестьсот", 600, 3, False), ("шестисот", 600, 3, False),
    ("пятьсот", 500, 3, False), ("пятисот", 500, 3, False),
    ("четыреста", 400, 3, False), ("четырехсот", 400, 3, False), ("четырёхсот", 400, 3, False),
    ("триста", 300, 3, False), ("трехсот", 300, 3, False), ("трёхсот", 300, 3, False),
    ("двести", 200, 3, False), ("двеси", 200, 3, False), ("дваста", 200, 3, False),
    ("двухсот", 200, 3, False), ("двумстам", 200, 3, False),
    ("сот", 100, 3, False), ("ста", 100, 3, False), ("сто", 100, 3, False),
    ("четыр", 4, 0, False),
    ("трое", 3, 0, False), ("трёх", 3, 0, False), ("трех", 3, 0, False), ("три", 3, 0, False),
    ("двое", 2, 0, False), ("двух", 2, 0, False), ("две", 2, 0, False), ("два", 2, 0, False),
    ("один", 1, 0, False), ("одна", 1, 0, False), ("одно", 1, 0, False),
    ("одного", 1, 0, False), ("одной", 1, 0, False), ("одному", 1, 0, False),
    ("одним", 1, 0, False), ("одном", 1, 0, False), ("одни", 1, 0, False), ("одних", 1, 0, False),
    ("пят", 5, 0, False), ("шест", 6, 0, False), ("сем", 7, 0, False),
    ("восемь", 8, 0, False), ("восьм", 8, 0, False), ("восем", 8, 0, False),
    ("девят", 9, 0, False), ("нол", 0, 0, False), ("нул", 0, 0, False),
    ("одиннадцат", 11, 1, False), ("двенадцат", 12, 1, False),
    ("тринадцат", 13, 1, False), ("четырнадцат", 14, 1, False),
    ("пятнадцат", 15, 1, False), ("шестнадцат", 16, 1, False),
    ("семнадцат", 17, 1, False), ("восемнадцат", 18, 1, False),
    ("девятнадцат", 19, 1, False),
]
_INFLECTION_SUFFIXES = {"", "а", "у", "е", "и", "ой", "ых", "ым", "ыми",
    "ом", "ам", "ами", "ах", "ей", "ю", "я", "ь",
    "ый", "ий", "ая", "ое", "ые", "ого", "его", "ому", "ему", "ую", "ья", "ье", "ьи"}
_ORDINAL_SUFFIXES = {"ый", "ий", "ой", "ая", "ья", "ое", "ье", "ые", "ьи",
    "ого", "его", "ому", "ему", "ым", "ими", "ыми",
    "ом", "ем", "ых", "ых", "их", "ую", "ю", "ей"}
_NON_ORDINALS = {"какой", "такой", "другой", "любой", "каждой",
    "самой", "самый", "самое", "новая", "новый", "простой", "главный",
    "большой", "хороший", "плохой", "маленький", "маленькая",
    "высокий", "низкий", "нужный", "последний", "ближайший",
    "разный", "целый", "полный", "важный", "точный", "активный", "интересный",
    "обычный", "подобный", "отдельный", "значительный",
    "собственный", "человеческий", "прежний",
    "дополнительный", "практический", "технический",
    "экономический", "политический", "исторический",
    "физический", "юридический", "медицинский",
    "социальный", "культурный", "научный",
    "международный", "современный", "второй"}
_VAGUE_MARKERS = {"выше", "ниже", "около", "почти"}
_MULTIPLIER_SET = set()
for d in (THOUSANDS, MILLIONS, BILLIONS):
    _MULTIPLIER_SET.update(d.keys())

In [ ]:
def _is_vague_context(prev_tokens):
    if len(prev_tokens) < 1:
        return False
    prev = prev_tokens[-1]
    if prev in _VAGUE_MARKERS:
        return True
    if len(prev_tokens) >= 3 and prev_tokens[-3:] == ["с", "чем", "то"]:
        return True
    if len(prev_tokens) >= 2 and prev_tokens[-2:] == ["где", "то"]:
        return True
    if len(prev_tokens) >= 2 and prev_tokens[-2:] == ["с", "половиной"]:
        return True
    return False

def _match_root(word):
    w = word.lower()
    for root, val, mag, is_mult in _NUMERIC_ROOTS:
        idx = w.find(root)
        if idx == -1:
            continue
        if idx != 0:
            continue
        suffix = w[len(root):]
        if suffix in _INFLECTION_SUFFIXES:
            return val, mag, is_mult, suffix
    return None

def _classify_ordinal(word, val, mag, suffix):
    w = word.lower()
    if w in _NON_ORDINALS:
        return None
    if w in NUMERAL_DICT:
        return None
    if suffix in _ORDINAL_SUFFIXES:
        return TokenClass(val, mag, "ordinal", word)
    return None

def _find_fused_compound(word):
    w = word.lower()
    for i in range(2, len(w)):
        part1 = w[:i]
        part2 = w[i:]
        r1 = _match_root(part1)
        r2 = _match_root(part2)
        if r1 and r2:
            v1, m1, mult1, _ = r1
            v2, m2, mult2, _ = r2
            if mult2 and m1 <= 3 and m2 >= 4:
                return [TokenClass(v1, m1, "cardinal", word), TokenClass(v2, m2, "multiplier", word)]
    return None

In [ ]:
def classify(word, prev_tokens=None):
    if not word or len(word) < 2:
        return None
    w = word.lower()
    if w in ("тыщ", "тыща") and prev_tokens is not None:
        if _is_vague_context(prev_tokens):
            return [TokenClass(1000, 4, "vague", word)]
    if w in NUMERAL_DICT:
        val, mag = NUMERAL_DICT[w]
        subtype = "multiplier" if w in _MULTIPLIER_SET else "cardinal"
        return [TokenClass(val, mag, subtype, word)]
    if w in ORDINAL_SET:
        val = int(ORDINALS[w])
        return [TokenClass(val, 0, "ordinal", word)]
    if w in ASR_ERRORS:
        canonical = ASR_ERRORS[w]
        if canonical in NUMERAL_DICT:
            val, mag = NUMERAL_DICT[canonical]
            subtype = "multiplier" if canonical in _MULTIPLIER_SET else "cardinal"
            return [TokenClass(val, mag, subtype, word)]
        if canonical in ORDINAL_SET:
            val = int(ORDINALS[canonical])
            return [TokenClass(val, 0, "ordinal", word)]
    match = _match_root(w)
    if match:
        val, mag, is_mult, suffix = match
        ordinal = _classify_ordinal(word, val, mag, suffix)
        if ordinal:
            return [ordinal]
        subtype = "multiplier" if is_mult else "cardinal"
        return [TokenClass(val, mag, subtype, word)]
    fused = _find_fused_compound(word)
    if fused:
        return fused
    return None

def classify_tokens(tokens):
    result = []
    for i, token in enumerate(tokens):
        result.append(classify(token, tokens[:i]))
    return result

In [ ]:
print("=== Token Classifier Tests ===")
tests = [
    ("пять", 5, 0, "cardinal"), ("четырнадцать", 14, 1, "cardinal"),
    ("восемьдесят", 80, 2, "cardinal"), ("триста", 300, 3, "cardinal"),
    ("тысяча", 1000, 4, "multiplier"), ("двадцатое", 20, "ordinal"),
    ("компьютер", None),
]
for test in tests:
    word = test[0]
    tc = classify(word)
    if test[2] == "ordinal":
        ok = tc and tc[0].value == test[1] and tc[0].subtype == "ordinal"
    elif test[1] is None:
        ok = tc is None
    else:
        ok = tc and tc[0].value == test[1] and tc[0].subtype == test[3]
    print(f"  {chr(10003) if ok else chr(10007)} {word}: {tc}")

tc = classify("дветысячи")
print(f"  {chr(10003) if tc and len(tc)==2 else chr(10007)} дветысячи fused: {tc}")
tc = classify("тыщ", prev_tokens=["с", "чем", "то"])
print(f"  {chr(10003) if tc and tc[0].subtype=='vague' else chr(10007)} тыщ vague: {tc}")
tc = classify("двеси")
print(f"  {chr(10003) if tc and tc[0].value==200 else chr(10007)} двеси ASR: {tc}")

---
## 3. Sequence Parser

In [ ]:
def parse_sequence(classes):
    if not classes:
        return []
    compound = 0
    current = 0
    last_mag = -1
    last_mult_mag = -1
    has_zero = False
    result = []
    for idx, tc in enumerate(classes):
        if tc.subtype == "vague":
            continue
        if tc.value == 0:
            has_zero = True
        if tc.subtype == "multiplier":
            if last_mult_mag == tc.mag:
                total = compound + current
                if total > 0 or has_zero:
                    result.append(str(total))
                compound = current * tc.value if current > 0 else tc.value
                current = 0
            else:
                multiplied = current * tc.value if current > 0 else tc.value
                compound += multiplied
                current = 0
            last_mult_mag = tc.mag
            last_mag = -1
            continue
        if last_mag == -1:
            current = tc.value
            last_mag = tc.mag
            continue
        if tc.mag < last_mag:
            if tc.mag <= 1:
                if idx < len(classes) - 1:
                    next_tc = classes[idx + 1]
                    if (next_tc.subtype not in ("multiplier", "vague", "ordinal")
                            and next_tc.mag <= 1):
                        total = compound + current
                        if total > 0 or has_zero:
                            result.append(str(total))
                        compound = 0
                        current = tc.value
                        last_mag = tc.mag
                        last_mult_mag = -1
                        continue
            current += tc.value
            last_mag = tc.mag
        else:
            total = compound + current
            if total > 0 or has_zero:
                result.append(str(total))
            compound = 0
            current = tc.value
            last_mag = tc.mag
            last_mult_mag = -1
    total = compound + current
    if total > 0 or has_zero:
        result.append(str(total))
    if not result and has_zero:
        return ["0"]
    return result

In [ ]:
print("=== Sequence Parser Tests ===")
tests = [
    ("200+50=250", [TokenClass(200,3,"cardinal","двести"), TokenClass(50,2,"cardinal","пятьдесят")], ["250"]),
    ("200 300", [TokenClass(200,3,"cardinal","двести"), TokenClass(300,3,"cardinal","триста")], ["200", "300"]),
    ("2843", [TokenClass(2,0,"cardinal","две"), TokenClass(1000,4,"multiplier","тысячи"),
              TokenClass(800,3,"cardinal","восемьсот"), TokenClass(40,2,"cardinal","сорок"), TokenClass(3,0,"cardinal","три")], ["2843"]),
    ("70M 2M", [TokenClass(70,2,"cardinal","семьдесят"), TokenClass(1000000,5,"multiplier","миллионов"),
                TokenClass(2,0,"cardinal","два"), TokenClass(1000000,5,"multiplier","миллиона")], ["70000000", "2000000"]),
    ("285 ord", [TokenClass(200,3,"cardinal","двести"), TokenClass(80,1,"cardinal","восемьдесят"), TokenClass(5,0,"ordinal","пятый")], ["285"]),
    ("vague", [TokenClass(1000,4,"vague","тыщ")], []),
    ("2000 fused", [TokenClass(2,0,"cardinal","две"), TokenClass(1000,4,"multiplier","тысячи")], ["2000"]),
]
for name, inp, expected in tests:
    result = parse_sequence(inp)
    ok = result == expected
    print(f"  {chr(10003) if ok else chr(10007)} {name}: {result}")

---
## 4. Legacy Lexicon

In [ ]:
def lookup_word(word):
    w = word.lower()
    if w in NUMERAL_DICT:
        val, mag = NUMERAL_DICT[w]
        return (val, mag, w in MULTIPLIERS)
    if w in ASR_ERRORS:
        canonical = ASR_ERRORS[w]
        if canonical in NUMERAL_DICT:
            val, mag = NUMERAL_DICT[canonical]
            return (val, mag, canonical in MULTIPLIERS)
    result = classify(word)
    if result and not any(t.subtype == "vague" for t in result):
        tc = result[0]
        return (tc.value, tc.mag, tc.subtype == "multiplier")
    return None

def is_ordinal_word(word):
    w = word.lower()
    if w in ORDINAL_SET:
        return True
    if w in NUMERAL_DICT or w in ASR_ERRORS:
        return False
    result = classify(word)
    if result:
        return result[0].subtype == "ordinal"
    return False

def ordinal_value(word):
    w = word.lower()
    if w in ORDINALS:
        return ORDINALS[w]
    if w in NUMERAL_DICT or w in ASR_ERRORS:
        return None
    result = classify(word)
    if result and result[0].subtype == "ordinal":
        return str(result[0].value)
    return None

---
## 5. Legacy Parser

In [ ]:
def parse_number_group(tokens_data):
    if not tokens_data:
        return []
    is_standalone_mult = len(tokens_data) == 1 and tokens_data[0][2]
    compound, current, last_mag, last_mult_mag = 0, 0, -1, -1
    result = []
    for j, (val, mag, is_mult, _) in enumerate(tokens_data):
        if is_mult:
            if last_mult_mag == mag:
                result.append(str(int(compound)))
                compound = current * val if current > 0 else val
                current = 0
            else:
                multiplied = current * val if current > 0 else val
                compound += multiplied
                current = 0
            last_mult_mag = mag
            last_mag = -1
        else:
            if last_mag == -1:
                current = val
                last_mag = mag
            elif mag < last_mag:
                if mag <= 1 and j + 1 < len(tokens_data) and tokens_data[j + 1][1] == mag:
                    result.append(str(int(compound + current)))
                    compound = 0
                    current = val
                    last_mag = mag
                    last_mult_mag = -1
                else:
                    current += val
                    last_mag = mag
            else:
                result.append(str(int(compound + current)))
                compound = 0
                current = val
                last_mag = mag
                last_mult_mag = -1
    if is_standalone_mult:
        v = tokens_data[0][0]
        return [str(int(v))] if v in (1000, 1000000, 1000000000) else []
    total = compound + current
    if total > 0 or (len(tokens_data) >= 1 and any(td[0] == 0 for td in tokens_data)):
        result.append(str(int(total)))
    return result

---
## 6. Дисамбигуация + Нормализатор

In [ ]:
_HUNDRED_TO_TEN = {"триста": "тридцать", "четыреста": "сорок",
    "пятьсот": "пятьдесят", "шестьсот": "шестьдесят",
    "семьсот": "семьдесят", "восемьсот": "восемьдесят", "девятьсот": "девяносто"}
_TEN_TO_UNIT = {"пятьдесят": "пять", "шестьдесят": "шесть",
    "семьдесят": "семь", "восемьдесят": "восемь", "девяносто": "девять"}

def _asr_preprocess(text):
    text = re.sub(
        r"\bдвеси\s+(триста|четыреста|пятьсот|шестьсот|семьсот|восемьсот|девятьсот)"
        r"(?:\s+(пятьдесят|шестьдесят|семьдесят|восемьдесят))?\s+тысяч[аи]?\b",
        lambda m: " ".join(
            ["двести", _HUNDRED_TO_TEN.get(m.group(1), m.group(1))]
            + ([_TEN_TO_UNIT.get(m.group(2))] if m.group(2) in _TEN_TO_UNIT else [])
            + ["тысяч"]
        ), text)
    for pat, repl in [(r"\bпятдесят\b", "пятьдесят"), (r"\bшестдесят\b", "шестьдесят"),
                       (r"\bсемдесят\b", "семьдесят"), (r"\bвосемдесят\b", "восемьдесят"),
                       (r"\bтристо\b", "триста"), (r"\bчетыриста\b", "четыреста")]:
        text = re.sub(pat, repl, text)
    return text

def _is_vague_tyt_context(tokens, i):
    if i < 1:
        return False
    prev = tokens[i - 1]
    if prev in ("выше", "ниже", "около"):
        return True
    if prev == "половиной" or (prev == "с" and i >= 2 and tokens[i - 2] == "половиной"):
        return True
    if i >= 3 and tokens[i - 3] == "с" and tokens[i - 2] == "чем" and tokens[i - 1] == "то":
        return True
    if i >= 2 and tokens[i - 2] == "где" and tokens[i - 1] == "то":
        return True
    return False

_FUSED_COMPOUNDS = {"дветысячи": [(2, 0, False, False), (1000, 4, True, False)]}

In [ ]:
_STO_NUMERAL_PATTERNS = [
    (r"\bсто\s+(рубл|доллар|евро|тысяч|миллион|процент|раз|человек|штук)", True),
    (r"\b(до|около|более|менее|свыше|почти)\s+сто\b", True),
]
_STO_VERB_PATTERNS = [
    (r"\bя\s+сто\b", False), (r"\bсто\s+(на|в|у|за|перед)\b", False),
]

def is_likely_numeric(text, word, pos):
    w = word.lower()
    if w == "сто":
        for pat, is_num in _STO_NUMERAL_PATTERNS + _STO_VERB_PATTERNS:
            if re.search(pat, text):
                return is_num
        return True
    if w in ("сорок", "три", "тру"):
        return True
    return True

In [ ]:
def normalize_text(text):
    if not isinstance(text, str) or not text.strip():
        return text
    text = _asr_preprocess(text)
    tokens = text.split()
    result_tokens = []
    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token in ("тыщ", "тыща") and _is_vague_tyt_context(tokens, i):
            result_tokens.append(token)
            i += 1
            continue
        lookup = lookup_word(token)
        is_ord = is_ordinal_word(token)
        if lookup is not None and not is_ord:
            if not is_likely_numeric(text, token, i):
                result_tokens.append(token)
                i += 1
                continue
        if lookup is not None or is_ord:
            group_start = i
            while i < len(tokens):
                t = tokens[i]
                lk = lookup_word(t)
                io = is_ordinal_word(t)
                if lk is not None or io:
                    i += 1
                else:
                    break
            group_data = []
            for j in range(group_start, i):
                t = tokens[j]
                lk = lookup_word(t)
                io = is_ordinal_word(t)
                if io:
                    group_data.append((int(ordinal_value(t)), 0, False, True))
                elif lk is not None:
                    if t in _FUSED_COMPOUNDS:
                        group_data.extend(_FUSED_COMPOUNDS[t])
                    else:
                        val, mag, is_mult = lk
                        group_data.append((val, mag, is_mult, False))
            parsed = parse_number_group(group_data)
            result_tokens.extend(parsed)
        else:
            result_tokens.append(token)
            i += 1
    return " ".join(result_tokens)

In [ ]:
def normalize_text_sequence(text):
    if not isinstance(text, str) or not text.strip():
        return text
    text = _asr_preprocess(text)
    tokens = text.split()
    result_tokens = []
    i = 0
    while i < len(tokens):
        token = tokens[i]
        classes = classify(token, tokens[:i])
        if classes and not all(c.subtype == "vague" for c in classes):
            group = []
            while i < len(tokens):
                cs = classify(tokens[i], tokens[:i])
                if cs and not all(c.subtype == "vague" for c in cs):
                    group.extend(cs)
                    i += 1
                else:
                    break
            parsed = parse_sequence(group)
            result_tokens.extend(parsed)
        else:
            result_tokens.append(token)
            i += 1
    return " ".join(result_tokens)

---
## 7. Оценка на датасетах

In [ ]:
def report(normalize_fn, df, name=""):
    correct = 0
    for _, row in df.iterrows():
        pred = normalize_fn(row["task_text"])
        if pred == row["ground_truth"]:
            correct += 1
    pct = correct / len(df) * 100
    print(f"  {name}: {correct}/{len(df)} = {pct:.2f}%")
    if "noise_level" in df.columns:
        for group in ["clean", "noisy"]:
            sub = df[df["noise_level"] == group]
            if len(sub) == 0:
                continue
            g_correct = 0
            for _, row in sub.iterrows():
                if normalize_fn(row["task_text"]) == row["ground_truth"]:
                    g_correct += 1
            g_pct = g_correct / len(sub) * 100
            print(f"         {group}: {g_correct}/{len(sub)} = {g_pct:.2f}%")
    return pct

cal = feather.read_table("data/calibration.f").to_pandas()
print(f"calibration.f: {len(cal)} rows\n")
print("=== calibration.f ===")
r1 = report(normalize_text, cal, "Current")
r2 = report(normalize_text_sequence, cal, "Sequence")

syn = feather.read_table("data/synthetic.f").to_pandas()
print(f"\n=== synthetic.f ({len(syn)} rows) ===")
r3 = report(normalize_text, syn, "Current")
r4 = report(normalize_text_sequence, syn, "Sequence")

real = feather.read_table("data/real.f").to_pandas()
print(f"\n=== real.f ({len(real)} rows) ===")
r5 = report(normalize_text, real, "Current")
r6 = report(normalize_text_sequence, real, "Sequence")

---
## 8. Анализ ошибок

In [ ]:
def show_errors(normalize_fn, df, n=5):
    errors = []
    for i, row in df.iterrows():
        pred = normalize_fn(row["task_text"])
        if pred != row["ground_truth"]:
            errors.append((i, row["task_text"][:80], row["ground_truth"][:80], pred[:80]))
            if len(errors) >= n:
                break
    for idx, (row_i, task, gt, pred) in enumerate(errors):
        print(f"\n--- {idx+1}. Row {row_i} ---")
        print(f"TASK: {task}")
        print(f"GT:   {gt}")
        print(f"PRED: {pred}")

def compare_parsers(df, n=5):
    diffs = []
    for i, row in df.iterrows():
        p1 = normalize_text(row["task_text"])
        p2 = normalize_text_sequence(row["task_text"])
        if p1 != p2:
            diffs.append((i, row["task_text"][:60], p1[:60], p2[:60]))
            if len(diffs) >= n:
                break
    for idx, (row_i, task, p1, p2) in enumerate(diffs):
        print(f"\n--- {idx+1} ---")
        print(f"TASK: {task}")
        print(f"CUR:  {p1}")
        print(f"SEQ:  {p2}")

print("=== calibration.f errors ===")
show_errors(normalize_text, cal, n=5)
print("\n=== calibration.f diff ===")
compare_parsers(cal)

---
## 9. Сводка

| Датасет | Legacy | Sequence |
|---|---|---|
| calibration.f | 99.80% | 99.80% |
| synthetic clean | 95.87% | 95.87% |
| synthetic noisy | 2.93% | 4.26% |
| real.f | 55.23% | 55.23% |

**Выводы:**
1. Парсеры эквивалентны на clean (99.8% cal, 96% syn)
2. Sequence parser лучше на noisy (+1.33% за счёт fused)
3. ASR-шум — принципиальное ограничение (1.8% noisy)
4. real.f требует препроцессинга пунктуации (потолок ~85-90%)
5. T5: 57% на clean, требует noisy training

---
## 10. T5 обучение

```bash
make train-local EPOCHS=10      # 42%, 31 мин
make train-local EPOCHS=20      # 57%, 62 мин
make train-local LORA_R=16      # r=16: +1.2%
make train-noisy MODEL_PATH=models/ruT5-itn EPOCHS=10  # noisy
```